<a href="https://colab.research.google.com/github/musishxo/house_price_analysis-SCB/blob/main/House_price_analysis_SCB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Price Analysis

## Syfte

Syftet med detta projekt är att analysera utvecklingen av bostadspriser i Sverige med hjälp av Python och öppna data från Statistiska centralbyrån (SCB).

Projektet kommer att undersöka hur bostadspriser har förändrats över tid och om det finns skillnader mellan olika svenska regioner.

## Frågeställning

**Hur har bostadspriserna utvecklats över tid i olika svenska regioner, och vilka skillnader kan identifieras mellan regionerna?**

## Metod

I projektet kommer data att hämtas från SCB:s öppna data. Datan kommer sedan att bearbetas och analyseras med Python.

Analysen kommer bland annat att innehålla:

* datarensning och förberedelse
* deskriptiv statistik
* jämförelse mellan regioner
* visualisering av prisutvecklingen
* beräkning av procentuell förändring
* analys av samband mellan variabler
* en enklare prediktiv analys

Resultaten kommer att visualiseras med hjälp av diagram och sammanfattas i en avslutande analys och reflektion.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**pandas** - läsa,struktera och analysera data
**numpy** - Matematiska beräkringar
**matplotlib** - skapa diagram

##  Datainsamling

I denna del kommer jag att använda Statistiska centralbyrån (SCB) som datakälla.

Jag kommer att hämta statistik om bostadspriser från SCB:s API. Datan innehåller information om bland annat genomsnittliga bostadspriser, antal försäljningar och geografiska områden i Sverige.



In [7]:
import requests

url = "https://statistikdatabasen.scb.se/api/v2/tables"

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    print("Anslutningen till SCB fungerar!")

except requests.exceptions.RequestException as error:
    print("Ett fel uppstod:", error)

Anslutningen till SCB fungerar!


requests.get() - skickar en förfrågan till SCB:s API.

response.raise_for_status()  - kontrollerar om servern returnerar ett HTTP-fel.

try/except - hanterar anslutningsfel utan att programmet avslutas oväntat.

In [ ]:
# Adress till tabellens metadata
metadata_url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3656/metadata"

try:
    response = requests.get(metadata_url, timeout=30)
    response.raise_for_status()

    metadata = response.json()

    print("Metadata har hämtats från SCB.")
    print("Antal variabler:", len(metadata["variables"]))

except requests.exceptions.RequestException as error:
    print("Ett fel uppstod vid hämtning av metadata:", error)

*KeyError: 'variables'* - Det betyder att "variables" inte finns som nyckel där vi förväntade oss. så vi ska undersöka hur API är strukterat.

In [9]:
# Undersök strukturen på svaret från SCB

print("Datatyp:", type(metadata))
print("Innehåll:")
print(metadata)

Datatyp: <class 'dict'>
Innehåll:
{'version': '2.0', 'class': 'dataset', 'label': 'Fastighetsprisstatistik för småhus (småhusbarometern), där månad avser en tremånadersperiod, se fotnot, efter region. Månad 1997M04-2026M08', 'source': 'SCB', 'updated': '2026-09-10T06:00:00Z', 'note': ['Uppgifterna från och med 2026 är preliminära.', 'Noteringen om månad avser i själva verket en tremånadersperiod, där noteringen avser den sista månaden i perioden. Alltså, för månadsnoteringen 2021M12 (december 2021) avser uppgifterna statistik för perioden oktober – december 2021.'], 'role': {'time': ['Tid'], 'metric': ['ContentsCode']}, 'id': ['Region', 'ContentsCode', 'Tid'], 'size': [25, 3, 353], 'dimension': {'Region': {'label': 'region', 'category': {'index': {'00': 0, '0010': 1, '0020': 2, '0030': 3, '01': 4, '03': 5, '04': 6, '05': 7, '06': 8, '07': 9, '08': 10, '09': 11, '10': 12, '12': 13, '13': 14, '14': 15, '17': 16, '18': 17, '19': 18, '20': 19, '21': 20, '22': 21, '23': 22, '24': 23, '25': 

Metadata visar att tabellen har tre variabler:

->Region – 25 regioner

->ContentsCode – 3 mått

->Tid – 353 tidsperioder

In [10]:
# Visa vilka variabler som finns i SCB:s dataset

for variable_code in metadata["id"]:
    variable = metadata["dimension"][variable_code]

    print("Kod:", variable_code)
    print("Namn:", variable["label"])
    print("Antal värden:", len(variable["category"]["index"]))
    print()

Kod: Region
Namn: region
Antal värden: 25

Kod: ContentsCode
Namn: tabellinnehåll
Antal värden: 3

Kod: Tid
Namn: månad
Antal värden: 353



+SCB har lagt informationen i "dimension" och inte "variables".

In [ ]:
data_url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3656/data"

# Hämta bostadsdtan : valt Sverige, medelpris och alla tidsperioder
query = {
    "selection": [
        {
            "variableCode": "Region",
            "valueCodes": ["00"]
        },
        {
            "variableCode": "ContentsCode",
            "valueCodes": ["BO0501AJ"]
        },
        {
            "variableCode": "Tid",
            "valueCodes": ["*"]
        }
    ]
}

try:
    response = requests.post(
        data_url,
        json=query,
        params={"outputFormat": "json-stat2"},
        timeout=30
    )

    response.raise_for_status()
    house_price_data = response.json()

    print("Bostadsdata har hämtats från SCB!")

except requests.exceptions.RequestException as error:
    print("Ett fel uppstod vid hämtning av data:", error)

## Hämtning av bostadsdata

Bostadsdata hämtas från SCB:s tabell.

I analysen används:
- Riket som region
- Medelpris för småhus som mått på bostadspris
- Alla tillgängliga tidsperioder

Resultatet innehåller 353 observationer.

In [12]:
print("Antal observationer:", len(house_price_data["value"]))
print("Första observationerna:")
print(house_price_data["value"][:10])

Antal observationer: 353
Första observationerna:
[731, 739, 748, 752, 758, 748, 738, 730, 739, 754]


## Spara bostadsdata

Efter att bostadsdata har hämtats från SCB sparas datan som en CSV-fil.
Det gör det möjligt att använda datasetet senare i projektet utan att behöva hämta datan från SCB varje gång.

In [24]:
# Skapa en DataFrame med bostadspriserna

house_prices = pd.DataFrame({
    "Tid": house_price_data["dimension"]["Tid"]["category"]["label"].values(),
    "Medelpris": house_price_data["value"]
})

house_prices.head(10)


CSV-filen har skapats!


+.head() - panda funktion som visar 5 rader som standard.


In [ ]:
house_prices.to_csv("house_prices.csv", index=False)

print("CSV-filen har skapats!")

## Undersöka datasetet

Innan datan bearbetas undersöks datasetets struktur och innehåll.
Vi kontrollerar bland annat antal rader och kolumner, datatyper samt om det finns saknade värden.

In [28]:
# Undersök datasetets struktur

print("Antal rader:", house_prices.shape[0])
print("Antal kolumner:", house_prices.shape[1])

print("\nKolumner:")
print(house_prices.columns.tolist())

print("\nDatatyper:")
print(house_prices.dtypes)

Antal rader: 353
Antal kolumner: 2

Kolumner:
['Tid', 'Medelpris']

Datatyper:
Tid          object
Medelpris     int64
dtype: object


In [29]:
# Kontrollera saknade värden

print("Saknade värden:")
print(house_prices.isna().sum())

Saknade värden:
Tid          0
Medelpris    0
dtype: int64


In [30]:
# Beskrivande statistik för medelpriset

house_prices["Medelpris"].describe()

,Medelpris
count,353.000000
mean,2257.932011
std,1028.502486
min,730.000000
25%,1331.000000
50%,2078.000000
75%,3042.000000
max,4168.000000
